# FloPy Intro Example

This notebook is a compact Day 3 starter that shows how to build a small MODFLOW 6 simulation directly in Python with `flopy`.

The workflow is:
1. import the FloPy tools used in the exercise
2. create a temporary working directory for model files
3. define a simple three-layer groundwater flow model
4. write head and budget output so the model can be inspected later

The example follows the official FloPy data tutorial and is meant to make the package structure visible before moving to the larger Gulf model notebook.

## Reference

This notebook was adapted from the FloPy MODFLOW 6 data tutorial:
`https://flopy.readthedocs.io/en/3.9.2/Notebooks/mf6_data_tutorial01.html`

In [2]:
# package import
from tempfile import TemporaryDirectory

## Import FloPy

`TemporaryDirectory` gives the model a disposable workspace, and `flopy` provides the MODFLOW 6 object model used throughout Day 3.

In [3]:
import flopy

## Create a Temporary Workspace

These variables define where model input files will be written and what the simulation will be named. Using a temporary directory keeps the example self-contained and easy to rerun.

In [4]:
temp_dir = TemporaryDirectory()
workspace = temp_dir.name
name = "tutorial01"

## Build the Simulation

This cell assembles a minimal MODFLOW 6 groundwater flow model:

- `MFSimulation` creates the overall simulation container
- `ModflowTdis` defines ten stress periods
- `ModflowIms` adds the iterative solver
- `ModflowGwf` creates the groundwater flow model
- `ModflowGwfdis`, `ModflowGwfic`, and `ModflowGwfnpf` define the grid, starting heads, and flow properties
- `ModflowGwfchd` adds constant-head boundaries
- `ModflowGwfoc` tells MODFLOW 6 to save head and budget output files

When the cell finishes, the simulation object exists in memory and is ready to be written or executed.

In [5]:
# set up simulation and basic packages
sim = flopy.mf6.MFSimulation(sim_name=name, sim_ws=workspace)
flopy.mf6.ModflowTdis(sim, nper=10, perioddata=[[365.0, 1, 1.0] for _ in range(10)])
flopy.mf6.ModflowIms(sim)
gwf = flopy.mf6.ModflowGwf(sim, modelname=name, save_flows=True)
botm = [30.0, 20.0, 10.0]
flopy.mf6.ModflowGwfdis(gwf, nlay=3, nrow=4, ncol=5, top=50.0, botm=botm)
flopy.mf6.ModflowGwfic(gwf)
flopy.mf6.ModflowGwfnpf(gwf, save_specific_discharge=True)
flopy.mf6.ModflowGwfchd(gwf, stress_period_data=[[(0, 0, 0), 1.0], [(2, 3, 4), 0.0]])
budget_file = f"{name}.bud"
head_file = f"{name}.hds"
flopy.mf6.ModflowGwfoc(
    gwf,
    budget_filerecord=budget_file,
    head_filerecord=head_file,
    saverecord=[("HEAD", "ALL"), ("BUDGET", "ALL")],
)
print("Done creating simulation.")

Done creating simulation.


## Next Step

To extend this example, add cells that call `sim.write_simulation()` and `sim.run_simulation()`, then inspect the generated head and budget files. The `Run_GULF_v4_TACC.ipynb` notebook shows that fuller workflow on a real model.